# MartiniSurf Colab: Protein + AutoMartini M3
### No-code workflow (forms only)

## 1) Install MartiniSurf

In [ ]:
#@title Install MartiniSurf (supports private repo via token)
REPO_URL = "https://github.com/jjimenezgar/MartiniSurf.git" #@param {type:"string"}
PRIVATE_REPO = True #@param {type:"boolean"}
INSTALL_VIEWER = True #@param {type:"boolean"}
ENSURE_MARTINIZE2 = True #@param {type:"boolean"}
ENSURE_DSSP_TOOL = False #@param {type:"boolean"}

import getpass
import shlex
import shutil
import subprocess
from pathlib import Path


def run_cmd(cmd, check=True):
    if isinstance(cmd, str):
        res = subprocess.run(cmd, shell=True, text=True, capture_output=True)
        shown = cmd
    else:
        res = subprocess.run(cmd, text=True, capture_output=True)
        shown = ' '.join(shlex.quote(x) for x in cmd)
    if check and res.returncode != 0:
        raise RuntimeError(f"Command failed ({shown})\\nSTDOUT:\\n{res.stdout[-4000:]}\\nSTDERR:\\n{res.stderr[-4000:]}")
    return res

run_cmd('rm -rf /content/MartiniSurf', check=False)

if PRIVATE_REPO:
    token = getpass.getpass('Paste your GitHub token (repo read access): ').strip()
    if not token:
        raise RuntimeError('Token is required for private repository installation.')
    if REPO_URL.startswith('https://'):
        clone_url = REPO_URL.replace('https://', f'https://{token}@', 1)
    else:
        raise RuntimeError('REPO_URL must use https://')
else:
    clone_url = REPO_URL

run_cmd(['git', 'clone', clone_url, '/content/MartiniSurf'])
run_cmd(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
run_cmd(['python', '-m', 'pip', 'install', '-q', '-e', '/content/MartiniSurf'])

if ENSURE_MARTINIZE2 and shutil.which('martinize2') is None:
    attempts = [
        ['python', '-m', 'pip', 'install', '-q', 'martinize2'],
        ['python', '-m', 'pip', 'install', '-q', 'vermouth'],
    ]
    for cmd in attempts:
        run_cmd(cmd, check=False)
        if shutil.which('martinize2') is not None:
            break

if ENSURE_DSSP_TOOL and shutil.which('mkdssp') is None:
    run_cmd(['apt-get', 'update', '-qq'], check=False)
    run_cmd(['apt-get', 'install', '-y', '-qq', 'dssp'], check=False)

if INSTALL_VIEWER:
    run_cmd(['python', '-m', 'pip', 'install', '-q', 'py3Dmol'])

print('Installed.')
print('martinisurf:', shutil.which('martinisurf') or 'NOT FOUND')
print('martinize2:', shutil.which('martinize2') or 'NOT FOUND')
print('mkdssp:', shutil.which('mkdssp') or 'NOT FOUND')


## 2) Optional linker generation (SMILES or uploaded file)

In [ ]:
#@title Optional linker generation (AutoMartini M3)
RUN_LINKER_GENERATION = False #@param {type:"boolean"}
INPUT_MODE = "SMILES" #@param ["SMILES", "Upload file"]
SMILES = "CCO" #@param {type:"string"}
MOLECULE_NAME = "LNK" #@param {type:"string"}
MOLECULE_CHARGE = 0 #@param {type:"integer"}

import shlex
import shutil
import subprocess
from pathlib import Path

if RUN_LINKER_GENERATION:
    out_dir = Path('/content/generated_linker_m3')
    out_dir.mkdir(parents=True, exist_ok=True)

    repo = Path('/content/Automartini_M3')
    subprocess.run(['rm', '-rf', str(repo)], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Martini-Force-Field-Initiative/Automartini_M3', str(repo)], check=True)

    req = repo / 'requirements.txt'
    if req.exists():
        subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', str(req)], check=False)
    subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(repo)], check=False)

    if INPUT_MODE == 'Upload file':
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No file uploaded.')
        infile = '/content/' + next(iter(uploaded.keys()))
        mode_args = {'input': infile}
    else:
        if not SMILES.strip():
            raise RuntimeError('SMILES is empty.')
        mode_args = {'smiles': SMILES.strip()}

    runner_specs = []
    for exe in ['automartini', 'auto_martini', 'AutoMartini']:
        if shutil.which(exe):
            runner_specs.append({'base': [exe], 'kind': 'exe', 'name': exe})

    for mod in ['automartini', 'auto_martini', 'AutoMartini']:
        probe = subprocess.run(['python', '-m', mod, '--help'], text=True, capture_output=True)
        if probe.returncode == 0:
            runner_specs.append({'base': ['python', '-m', mod], 'kind': 'module', 'name': mod})

    for script in repo.rglob('*.py'):
        n = script.name.lower()
        if ('martini' in n) and ('auto' in n or 'map' in n):
            runner_specs.append({'base': ['python', str(script.resolve())], 'kind': 'script', 'name': script.name})

    if not runner_specs:
        raise RuntimeError('No AutoMartini runner found after installation.')

    def pick_flag(help_text, options, default):
        for opt in options:
            if opt in help_text:
                return opt
        return default

    tried = []
    ok = False
    for spec in runner_specs:
        if isinstance(spec['base'], list) and len(spec['base']) == 3 and spec['base'][0] == 'python' and spec['base'][1] == '-m':
            cmd_base = spec['base']
        else:
            cmd_base = spec['base']

        help_cmd = cmd_base + ['--help']
        help_run = subprocess.run(help_cmd, text=True, capture_output=True)
        if help_run.returncode != 0:
            help_text = ''
        else:
            help_text = (help_run.stdout or '') + '\n' + (help_run.stderr or '')

        smi_flag = pick_flag(help_text, ['--smi', '--smiles'], '--smi')
        in_flag = pick_flag(help_text, ['--mol', '--input', '-i'], '--input')
        name_flag = pick_flag(help_text, ['--molname', '--name'], '--molname')
        out_flag = pick_flag(help_text, ['--output', '--out', '-o'], '--output')
        charge_flag = pick_flag(help_text, ['--charge', '-c'], '--charge')

        cmd = list(cmd_base)
        if 'smiles' in mode_args:
            cmd += [smi_flag, mode_args['smiles']]
        else:
            cmd += [in_flag, mode_args['input']]

        cmd += [name_flag, MOLECULE_NAME, charge_flag, str(MOLECULE_CHARGE), out_flag, str(out_dir)]
        tried.append(' '.join(shlex.quote(x) for x in cmd))
        print('Trying:', tried[-1])

        run = subprocess.run(cmd, text=True, capture_output=True)
        if run.returncode == 0:
            ok = True
            break

    if not ok:
        print('Attempted commands:')
        for item in tried[-12:]:
            print(' -', item)
        raise RuntimeError('AutoMartini M3 failed with all detected runners.')

    print('Done. Generated files:')
    subprocess.run(['ls', '-lh', str(out_dir)], check=False)
else:
    print('Skipped.')


## 3) Build system

In [ ]:
#@title Build protein system
PDB_INPUT_MODE = "PDB ID" #@param ["PDB ID", "Upload PDB file"]
PDB_ID = "1RJW" #@param {type:"string"}
MOLTYPE = "Protein" #@param {type:"string"}
USE_DSSP = False #@param {type:"boolean"}
OUTDIR = "Simulation_Files" #@param {type:"string"}

USE_EXISTING_SURFACE = False #@param {type:"boolean"}
SURFACE_SOURCE = "Upload surface.gro now" #@param ["Upload surface.gro now", "Use existing surface.gro in session"]
SURFACE_FILE = "surface.gro" #@param {type:"string"}
LX = 20.0 #@param {type:"number"}
LY = 20.0 #@param {type:"number"}
SURFACE_BEAD = "C1" #@param {type:"string"}

USE_LINKER = False #@param {type:"boolean"}
LINKER_SOURCE = "Upload linker files now" #@param ["Upload linker files now", "Use existing linker.gro/.itp in session"]
LINKER_GRO = "linker.gro" #@param {type:"string"}
INVERT_LINKER = False #@param {type:"boolean"}

ANCHOR_1 = "1 8 10 11" #@param {type:"string"}
ANCHOR_2 = "2 1025 1027 1028" #@param {type:"string"}
DIST = 10.0 #@param {type:"number"}
DRY_RUN = False #@param {type:"boolean"}

import shlex
import shutil
import subprocess
from pathlib import Path

if shutil.which('martinisurf') is None:
    raise RuntimeError('martinisurf is not installed. Run the Install cell first.')
if shutil.which('martinize2') is None:
    raise RuntimeError('martinize2 is not installed. Re-run Install with ENSURE_MARTINIZE2=True.')
if USE_DSSP and shutil.which('mkdssp') is None:
    raise RuntimeError('USE_DSSP=True but mkdssp is missing. Re-run Install with ENSURE_DSSP_TOOL=True or disable DSSP.')

if PDB_INPUT_MODE == 'Upload PDB file':
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No PDB uploaded')
    pdb_input = '/content/' + next(iter(uploaded.keys()))
else:
    pdb_input = PDB_ID.strip()

if USE_EXISTING_SURFACE:
    if SURFACE_SOURCE == 'Upload surface.gro now':
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No surface file uploaded')
        surface_path = '/content/' + next(iter(uploaded.keys()))
    else:
        surface_path = '/content/' + SURFACE_FILE
        if not Path(surface_path).exists():
            raise FileNotFoundError(f'Missing file: {surface_path}')

cmd = ['martinisurf', '--pdb', pdb_input, '--moltype', MOLTYPE, '--outdir', OUTDIR]
if not USE_DSSP:
    cmd += ['--no-dssp']

if USE_EXISTING_SURFACE:
    cmd += ['--surface', surface_path]
else:
    cmd += ['--lx', str(LX), '--ly', str(LY), '--surface-bead', SURFACE_BEAD]

if USE_LINKER:
    if LINKER_SOURCE == 'Upload linker files now':
        from google.colab import files
        print('Upload linker.gro and linker.itp (matching basename preferred)')
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No linker files uploaded')
        names = list(uploaded.keys())
        gros = [n for n in names if n.lower().endswith('.gro')]
        itps = [n for n in names if n.lower().endswith('.itp')]
        if not gros:
            raise RuntimeError('Please upload one linker .gro file')
        if not itps:
            raise RuntimeError('Please upload one linker .itp file')
        linker_gro = Path('/content') / gros[0]
        linker_itp = Path('/content') / itps[0]
        target_itp = linker_gro.with_suffix('.itp')
        if linker_itp != target_itp:
            shutil.copyfile(linker_itp, target_itp)
        linker_path = str(linker_gro)
    else:
        linker_path = '/content/' + LINKER_GRO
        if not Path(linker_path).exists():
            raise FileNotFoundError(f'Missing linker file: {linker_path}')
        expected_itp = Path(linker_path).with_suffix('.itp')
        if not expected_itp.exists():
            raise FileNotFoundError(f'Missing linker topology: {expected_itp}')

    cmd += ['--linker', linker_path]
    if INVERT_LINKER:
        cmd += ['--invert-linker']
    cmd += ['--linker-group'] + ANCHOR_1.split()
else:
    cmd += ['--anchor'] + ANCHOR_1.split()
    if ANCHOR_2.strip():
        cmd += ['--anchor'] + ANCHOR_2.split()
    cmd += ['--dist', str(DIST)]

print('Running:\n' + ' '.join(shlex.quote(x) for x in cmd))
if DRY_RUN:
    print('DRY_RUN=True, command not executed.')
else:
    res = subprocess.run(cmd, text=True, capture_output=True)
    print(res.stdout[-12000:])
    if res.returncode != 0:
        print(res.stderr[-12000:])
        combined = ((res.stdout or '') + '\n' + (res.stderr or '')).lower()
        msg = 'MartiniSurf failed. '
        if 'dssp' in combined and ('failed' in combined or 'not found' in combined):
            msg += 'Likely cause: DSSP executable issue. Disable DSSP or install mkdssp in Install cell.'
        elif 'martinize2' in combined and ('not found' in combined or 'no such file' in combined):
            msg += 'Likely cause: martinize2 missing in PATH.'
        elif 'failed to download rcsb' in combined:
            msg += 'Likely cause: network issue or invalid PDB ID.'
        raise RuntimeError(msg)

    print('Build completed')


## 4) Download

In [ ]:
#@title Download results as ZIP
OUTPUT_FOLDER = "Simulation_Files" #@param {type:"string"}
ZIP_NAME = "Simulation_Files" #@param {type:"string"}

import shutil
from pathlib import Path
from google.colab import files

folder = Path('/content') / OUTPUT_FOLDER
if not folder.exists():
    raise FileNotFoundError(f'Folder not found: {folder}')

shutil.make_archive(ZIP_NAME, 'zip', str(folder))
files.download(ZIP_NAME + '.zip')


## 5) Visualize final structure

In [ ]:
#@title Viewer (generated structure)
SYSTEM_GRO_FILE = "Simulation_Files/2_system/immobilized_system.gro" #@param {type:"string"}
STYLE = "Spheres" #@param ["Spheres", "Sticks"]

import py3Dmol
from pathlib import Path

path = Path('/content') / SYSTEM_GRO_FILE
if not path.exists():
    raise FileNotFoundError(f'File not found: {path}')

model = path.read_text()
view = py3Dmol.view(width='100%', height=800)
view.addModel(model, 'gro')
if STYLE == 'Sticks':
    view.setStyle({'stick': {}})
else:
    view.setStyle({'sphere': {'radius': 1.4}})
view.zoomTo()
view.show()


## Acknowledgements and Licenses
This notebook uses third-party tools and libraries with their own licenses:

- **martinize2** (Martini Force Field Initiative, Apache-2.0)
- **AutoMartini M3** (Martini Force Field Initiative, see repository license)
- **MartiniSurf** (this repository)
- Python ecosystem packages used here, including (depending on step): `numpy`, `scipy`, `MDAnalysis`, `mdtraj`, `py3Dmol`

Please cite and comply with each upstream project license when publishing results.
